In [ ]:
# В данном разделе мы научимся:
# - Создавать 5 вопросов для одного документа


In [3]:
from ingest import load_faq_data
documents = load_faq_data()

In [ ]:
# Будем генерировать вопросы только из курса LLM на Zoomcamp
# так как если будем из всего курса выйдет дорожен

documents_llm = []

for doc in documents:
  if doc["course"] == "llm-zoomcamp":
    documents_llm.append(doc)

len(documents_llm)

# 153 вопросов - именно на этих вопросах мы будем тестить 
# оценку работы нашего ассистента

153

In [5]:
documents = documents_llm

In [ ]:
doc = documents[0]

print(doc["id"])        
# => 74eb249bbf

print(doc["question"])  
# => I just discovered the course. Can I still join?

print(doc["answer"])    
# => Yes, but if you want to receive a certificate, you need 
# to submit your project while we’re still accepting submissions.

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [ ]:
# Id становится меткой в ​​нашем истинном наборе данных. Мы генерируем вопросы на
# основе документа, поэтому знаем, что этот документ содержит ответ. Позже, 
# при оценке результатов поиска, проверяется, возвращает ли поиск документ с этим id.

# Именно поэтому каждой записи необходим неизменяемый id. Если вы не сможете найти ID, 
# то вы не сможете узнать, нашел ли поиск нужный документ. При создании собственного 
# набора данных для оценки сначала присвойте id каждой записи в вашей базе знаний.

In [13]:
# Генерация вопросов с помощью структурированного вывода (выводить данные в 
# структурированной форме). Создаем класс, который будет следовать схеме - в 
# конце мы получим объект с полями строк

from pydantic import BaseModel

class Questions(BaseModel):
  questions: list[str]


In [7]:
data_gen_instructions = """
  You emulate a student who's taking our course.
  Formulate 5 questions this student might ask based on a FAQ record. The record
  should contain the answer to the questions, and the questions should be complete and not too short.
  If possible, use as fewer words as possible from the record.

  The output should resemble how people ask questions
  on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
# Подготавливаем документ в формате JSON:
import json

user_prompt = json.dumps(doc)

'''
{
 "id": "74eb249bbf"
 "course": "llm-zoomcamp"
 "section": "General Course-Related Questions", 
 "question": "I just discovered the course. Can I still join?", 
 "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.  
}
'''

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [ ]:
# И создаем сообщение
messages = [
  {"role": "developer", "content": data_gen_instructions},
  {"role": "user", "content": user_prompt}
]

In [ ]:
# Для структурированного вывода мы переключаемся на responses.parse 
# и передаем text_format=Questions, что указывает API возвращать 
# наш класс вместо свободного текста.

response = openai_client.responses.parse(
  model = "gpt-5.4-mini",
  input = messages,
  text_format = Questions
)

# Вот наш ответ - он же содержит вопросы, которые могут задать 
# пользователи (cгенерировали)
response.output_parsed.questions

'''
[
 'I just found this course — can I still join it now?',
 'If I start the course late, am I still allowed to participate?',
 'Is it too late to join this course after it has already started?',
 'Can I still get a certificate if I join the course now?',
 'What do I need to do to be eligible for the certificate if I’m joining late?'
'''

['I just found this course — can I still join it now?',
 'If I start the course late, am I still allowed to participate?',
 'Is it too late to join this course after it has already started?',
 'Can I still get a certificate if I join the course now?',
 'What do I need to do to be eligible for the certificate if I’m joining late?']

In [ ]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
  openai_client,
  data_gen_instructions,
  user_prompt,
  Questions
)

print(result.questions)

['I found this course late — can I still enroll and follow along?', 'Am I allowed to join after the course has already started?', 'If I join now, is it still possible to get a certificate?', "What do I need to do to qualify for the certificate if I'm starting late?", 'Is there still time to submit the project for a certificate?']


In [18]:
# Сколько входных и выходных токенов было потрачено

usage.input_tokens, usage.output_tokens

(212, 84)

In [ ]:
# А теперь рассчитываем цену на основе response.usage

from evaluation_utils import calc_price

cost = calc_price(usage)
cost

'''
  {
    'input_cost': 0.00015900000000000002,
    'output_cost': 0.00037799999999999997,
    'total_cost': 0.0005369999999999999
  }
'''

{'input_cost': 0.00015900000000000002,
 'output_cost': 0.00037799999999999997,
 'total_cost': 0.0005369999999999999}

In [ ]:
# Теперь преобразуйте эти вопросы в настоящие данные:

records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

'''
[
 {
  'question': 'I found this course late — can I still enroll and follow along?',
  'document': '74eb249bbf'
 },
 {
  'question': 'Am I allowed to join after the course has already started?',
  'document': '74eb249bbf'
 },
 {
  'question': 'If I join now, is it still possible to get a certificate?',
  'document': '74eb249bbf'
 },
 {
  'question': "What do I need to do to qualify for the certificate if I'm starting late?",
  'document': '74eb249bbf'
 },
 {
  'question': 'Is there still time to submit the project for a certificate?',
  'document': '74eb249bbf'
 }
]
'''

[{'question': 'I found this course late — can I still enroll and follow along?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to join after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, is it still possible to get a certificate?',
  'document': '74eb249bbf'},
 {'question': "What do I need to do to qualify for the certificate if I'm starting late?",
  'document': '74eb249bbf'},
 {'question': 'Is there still time to submit the project for a certificate?',
  'document': '74eb249bbf'}]

In [ ]:
# Теперь мы знаем, как создавать и сохранять вопросы для одного 
# документа. На следующем уроке мы проделаем это для всех документов
# с часто задаваемыми вопросами (FAQ) по программе LLM на Zoomcamp
# и сохраним полный набор эталонных данных.